# Lab 4, Silver customers (SCD Type 2)

This builds a proper SCD Type 2 table out of the bronze customers data. I'm deliberately not using the source's own valid_from and valid_to columns, I'm treating every load as a brand new snapshot and tracking history myself with effective_start, effective_end and is_current columns. Feels more realistic than trusting a source to have already done the change tracking for me.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "lab4", "Catalog")
catalog = dbutils.widgets.get("catalog")

## Casting bronze strings to proper types, with a quarantine for anything that doesn't fit

In [0]:
cust_bronze = spark.table(f"{catalog}.bronze.brz_customers")

cust_typed = cust_bronze.select(
    F.expr("try_cast(customer_id AS INT)").alias("customer_id"),
    F.expr("try_cast(tax_id AS DOUBLE)").alias("tax_id"),
    F.col("tax_code"),
    F.col("customer_name"),
    F.col("state"),
    F.col("city"),
    F.col("postcode"),
    F.col("street"),
    F.col("number"),
    F.col("unit"),
    F.col("region"),
    F.col("district"),
    F.expr("try_cast(lon AS DOUBLE)").alias("lon"),
    F.expr("try_cast(lat AS DOUBLE)").alias("lat"),
    F.col("ship_to_address"),
    F.expr("try_cast(valid_from AS INT)").alias("valid_from"),
    F.expr("try_cast(valid_to AS DOUBLE)").alias("valid_to"),
    F.expr("try_cast(units_purchased AS DOUBLE)").alias("units_purchased"),
    F.expr("try_cast(loyalty_segment AS INT)").alias("loyalty_segment"),
    F.col("customer_id").alias("raw_customer_id"),  # kept for the quarantine table only
)

# customer_id is the merge key
cust_quarantine = (
    cust_typed
    .filter("customer_id IS NULL")
    .withColumn("rejection_reason", F.lit("customer_id did not cast to INT"))
    .withColumn("quarantined_at", F.current_timestamp())
)

cust_clean = cust_typed.filter("customer_id IS NOT NULL").drop("raw_customer_id")

quarantine_count = cust_quarantine.count()
print(f"{quarantine_count} row(s) failed the customer_id cast and were quarantined")

if quarantine_count > 0:
    (cust_quarantine.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{catalog}.silver.slv_customers_quarantine")
    )

## Deduping the bronze snapshot

There were 143 duplicate rows in bronze (checked pre-cast, so this count is unaffected by the casting/quarantine step above). For the tiebreak I'm just keeping whichever row has the higher units_purchased, treating that as the more complete looking record. row_number() over a window partitioned by customer_id does the job, keeping row 1 per customer. Note this now runs against `cust_clean` (post-cast) rather than the raw bronze table directly, since `units_purchased` needs to actually be a DOUBLE for `F.desc(...)` to sort it correctly - sorting it as a string would tiebreak on lexical order, not real values.

In [0]:
from pyspark.sql.window import Window

cust_window = Window.partitionBy("customer_id").orderBy(F.desc("units_purchased"))

cust_dedup = (
    cust_clean
    .withColumn("row_num", F.row_number().over(cust_window))
    .filter("row_num = 1")
    .drop("row_num")
)

print(f"Deduped: {cust_dedup.count()} rows (was 28,813 in bronze)")

## Creating the silver table

Same business columns as the source, plus four columns I own myself: effective_start (when this version became current), effective_end (null while current), is_current (so I can just filter for today's data without a subquery), and silver_updated_at as an audit column.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS lab4.silver.slv_customers_scd2 (
    customer_id INT,
    tax_id DOUBLE,
    tax_code STRING,
    customer_name STRING,
    state STRING,
    city STRING,
    postcode STRING,
    street STRING,
    number STRING,
    unit STRING,
    region STRING,
    district STRING,
    lon DOUBLE,
    lat DOUBLE,
    ship_to_address STRING,
    valid_from INT,
    valid_to DOUBLE,
    units_purchased DOUBLE,
    loyalty_segment INT,
    effective_start TIMESTAMP,
    effective_end TIMESTAMP,
    is_current BOOLEAN,
    silver_updated_at TIMESTAMP
)
USING DELTA

## The merge function

One thing worth noting: this runs in two passes. A single MERGE can't close an old row and insert its replacement in the same pass against the same key, so the second part (opening the new current row) runs as its own follow up insert.

In [0]:
def run_scd2_merge(source_df, target_table_name):
    target = DeltaTable.forName(spark, target_table_name)

    tracked_cols = [
        "tax_id", "tax_code", "customer_name", "state", "city", "postcode",
        "street", "number", "unit", "region", "district", "lon", "lat",
        "ship_to_address", "units_purchased", "loyalty_segment",
    ]

    # anything that actually differs from the current row counts as a real change
    change_condition = " OR ".join([f"target.{c} <> source.{c}" for c in tracked_cols])

    source_staged = source_df.withColumn("merge_key", F.col("customer_id"))

    (target.alias("target")
        .merge(
            source_staged.alias("source"),
            "target.customer_id = source.merge_key AND target.is_current = true",
        )
        .whenMatchedUpdate(
            condition=change_condition,
            set={
                "is_current": "false",
                "effective_end": "current_timestamp()",
                "silver_updated_at": "current_timestamp()",
            },
        )
        .whenNotMatchedInsert(
            values={
                **{c: f"source.{c}" for c in ["customer_id", *tracked_cols, "valid_from", "valid_to"]},
                "effective_start": "current_timestamp()",
                "effective_end": "CAST(NULL AS TIMESTAMP)",
                "is_current": "true",
                "silver_updated_at": "current_timestamp()",
            }
        )
        .execute()
    )

    # anyone we just closed out above needs a new current row opened for them
    closed_customers = source_df.join(
        spark.table(target_table_name).filter("is_current = false").select("customer_id").distinct(),
        on="customer_id",
        how="inner",
    )

    if closed_customers.count() > 0:
        current_now = spark.table(target_table_name).filter("is_current = true").select("customer_id")
        rows_to_reopen = closed_customers.join(current_now, on="customer_id", how="left_anti")

        (rows_to_reopen
            .withColumn("effective_start", F.current_timestamp())
            .withColumn("effective_end", F.lit(None).cast("timestamp"))
            .withColumn("is_current", F.lit(True))
            .withColumn("silver_updated_at", F.current_timestamp())
            .write.format("delta").mode("append").saveAsTable(target_table_name)
        )


run_scd2_merge(cust_dedup, f"{catalog}.silver.slv_customers_scd2")

spark.sql(f"SELECT COUNT(*) AS total_rows, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows FROM {catalog}.silver.slv_customers_scd2").show()

## Faking a second snapshot

The source file never changes on its own, so to actually see SCD2 do something I need to manufacture a change myself. I picked 25 random customers and bumped their loyalty tier and set their state to "ZZ", just so it's obvious in later queries which rows I deliberately touched.

In [0]:
sample_ids = cust_dedup.select("customer_id").limit(25)

changed_cust = (
    cust_dedup
    .join(sample_ids, on="customer_id", how="inner")
    .withColumn("loyalty_segment", (F.col("loyalty_segment") + 1) % 4)
    .withColumn("state", F.lit("ZZ"))
)

unchanged_cust = cust_dedup.join(sample_ids, on="customer_id", how="left_anti")

day2_snapshot = unchanged_cust.unionByName(changed_cust)

print(f"Simulated day 2 snapshot: {day2_snapshot.count()} rows, 25 deliberately changed")

## Running the merge again with the changed snapshot

This is the actual proof it works. The 25 changed customers should now show up twice, once as a closed out history row and once as the new current one. Everyone else should be untouched.

In [0]:
run_scd2_merge(day2_snapshot, f"{catalog}.silver.slv_customers_scd2")

spark.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows,
           SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS historical_rows
    FROM {catalog}.silver.slv_customers_scd2
""").show()

Checking one of the changed customers directly, this is the screenshot I actually want for the lab.

In [0]:
%sql
SELECT customer_id, loyalty_segment, state, effective_start, effective_end, is_current
FROM lab4.silver.slv_customers_scd2
WHERE customer_id IN (SELECT customer_id FROM lab4.silver.slv_customers_scd2 WHERE state = 'ZZ')
ORDER BY customer_id, effective_start

## Idempotency check

Running the exact same day 2 snapshot through the merge again. Row count before and after should be identical, that's what proves reruns don't create duplicate history.

In [0]:
before_count = spark.table(f"{catalog}.silver.slv_customers_scd2").count()

run_scd2_merge(day2_snapshot, f"{catalog}.silver.slv_customers_scd2")

after_count = spark.table(f"{catalog}.silver.slv_customers_scd2").count()

print(f"Before rerun: {before_count} rows")
print(f"After rerun: {after_count} rows")
print("Idempotent, good" if before_count == after_count else "Not idempotent, need to check this")